# 31. Microsoft GraphRAG (Global Search)

Detect communities in the entity graph, summarise each, and answer global questions with map-reduce.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Vector RAG answers **local** questions (a fact in one chunk) but fails on **global** ones: *'What are the main themes of this corpus?'* - no single chunk contains the answer. **Microsoft GraphRAG** builds an entity graph, detects **communities** (clusters of closely related entities), writes a **report** for each, and answers global questions with **map-reduce** over those reports.

## How it works

1. Extract entities/relations -> graph
2. Detect communities (Leiden in the paper; Louvain here)
3. LLM writes a report per community
4. **Map:** each report answers the question and rates its relevance
5. **Reduce:** combine the best partial answers into a final answer

```text
 graph --> communities C1..Cn --> report R1..Rn
 global question --> [map: each Ri answers + score] --> keep high scores --> [reduce] --> final answer
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install networkx`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.Graph()
for i, chunk in enumerate(chunks):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r)
print(G.number_of_nodes(), "entities,", G.number_of_edges(), "edges")

## Community detection
Microsoft uses Leiden; networkx ships Louvain, which behaves similarly for this demo.

In [ ]:
communities = nx.community.louvain_communities(G, seed=42)
communities = [c for c in communities if len(c) >= 3]
print(len(communities), "communities")

## Community reports
Each community's facts are summarised once, at index time.

In [ ]:
reports = []
for c in communities:
    facts = [f"{u} {d['relation']} {v}" for u, v, d in G.subgraph(c).edges(data=True)]
    reports.append(llm("Write a 2-sentence report describing the theme of these facts:\n" + "\n".join(facts)))
for r in reports[:3]: print("-", r)

## Global search = map-reduce
**Map:** each report answers the question with a helpfulness score. **Reduce:** combine the best partial answers.

In [ ]:
def global_search(query):
    partials = []
    for rep in reports:
        out = llm_json(f'Using only this report, answer the question and rate relevance 0-10. Reply as JSON {{"answer": str, "score": int}}.\n\nReport: {rep}\n\nQuestion: {query}')
        if isinstance(out, dict) and out.get("score", 0) >= 5:
            partials.append(out["answer"])
    return llm("Combine these partial answers into one coherent answer:\n- " + "\n- ".join(partials)) if partials else "No relevant communities."

print(global_search("What are the main themes of this handbook?"))

## Inspect the communities

In [ ]:
for i, c in enumerate(communities[:5]):
    print(f"community {i} ({len(c)} entities): {sorted(c)[:8]}")
    print("   report:", reports[i][:200], "\n")

## Visualise communities by colour

In [ ]:
import matplotlib.pyplot as plt
colors = {}
for i, c in enumerate(communities):
    for n in c: colors[n] = i
plt.figure(figsize=(10, 7))
sub = G.subgraph([n for n in G if n in colors])
pos = nx.spring_layout(sub, seed=5, k=0.8)
nx.draw_networkx(sub, pos, node_color=[colors[n] for n in sub], cmap="tab20", node_size=260, font_size=6, edge_color="#bbb")
plt.axis("off"); plt.tight_layout(); plt.show()

## Local vs global questions
Compare vector RAG (good at local) with global search (good at themes).

In [ ]:
global_q = "What are the main themes and topics covered across the whole handbook?"
print("VECTOR RAG (top-3 chunks):\n", rag(global_q, store, 3)[0], "\n")
print("GLOBAL SEARCH (community reports):\n", global_search(global_q))

In [ ]:
local_q = "What is the Reciprocal Rank Fusion constant?"
print("VECTOR RAG:", rag(local_q, store, 3)[0])
print("GLOBAL SEARCH:", global_search(local_q))

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Answers corpus-wide questions vector RAG cannot | Expensive indexing (extraction + reports) |
| Community reports are reusable summaries | Poor at precise facts |
| Scales by parallelising the map step | Quality depends on extraction and community quality |

## When to use it

Sense-making over large corpora: themes, trends, summaries across documents. Use vector search for lookup questions and route between them.

## Experiments to try

- Build hierarchical communities (communities of communities).
- Route automatically between local and global search.
- Cache the reports on disk.

## Key takeaways

- Communities summarise structure.
- Map-reduce answers global questions.
- Use both local and global search.

---
**Next:** [32. LightRAG](../32_light_rag/light_rag.ipynb)